## CYS5550: Introduction to Artificial Intelligence
## Week 6 Homework: Local LLM NLP Project

### Assignment Overview

| Item | Details |
|------|---------|
| **Course** | CYS5550 – Introduction to Artificial Intelligence |
| **Assignment** | Week 6 Homework |
| **Due Date** | Before Week 7 Lecture |
| **Submission** | Submit Github URL of the file |

### Instructions

Build an NLP application using a **LOCAL LLM** for **ONE** of the following tasks:

### Option A: Customer Support Classifier
- Classify support tickets into categories (Technical, Billing, General, Complaint)
- Extract key information (product, issue type)
- Suggest response templates
- Test on at least 15 example tickets

### Option B: News Article Analyzer
- Summarize articles
- Extract named entities
- Classify topic (politics, sports, tech, etc.)
- Test on at least 5 news articles

### Option C: Recipe Assistant
- Parse ingredients from text
- Scale quantities for different servings
- Suggest substitutions
- Answer cooking questions

### Requirements

1. Use a **LOCAL LLM** (TinyLlama, Phi-2, or Llama-2 quantized)
2. Include at least **2 different NLP tasks**
3. Compare LLM results with at least **one other method** (rules or ML)
4. Document **prompt engineering choices**
5. Provide **test cases** with expected vs actual outputs

### Part 1: Setup and Model Loading

Run the cells below to set up your environment.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install transformers torch accelerate -q
# !pip install bitsandbytes -q  # For quantization (Linux/Colab)
# !pip install sentencepiece -q
# !pip install nltk spacy scikit-learn -q

import warnings
warnings.filterwarnings('ignore')

print("Setup complete!")

In [ ]:
# System check
import torch

print("="*50)
print("SYSTEM CHECK")
print("="*50)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {gpu_mem:.1f} GB")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print("Apple MPS available")
else:
    print("Running on CPU - TinyLlama recommended")
print("="*50)

In [ ]:
# Load Local LLM
from transformers import pipeline
import torch

print("Loading TinyLlama-1.1B-Chat...")
print("(First run downloads ~2GB)")

generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16,
    device_map="auto"
)

print("✓ Model loaded successfully!")

In [ ]:
# Helper function for text generation
def generate_response(prompt, max_tokens=150, temperature=0.3):
    """
    Generate response from local LLM.
    
    Args:
        prompt: The prompt text
        max_tokens: Maximum tokens to generate
        temperature: Creativity (0.1-1.5)
    
    Returns:
        Generated text string
    """
    chat_prompt = f"""<|system|>
You are a helpful AI assistant.</s>
<|user|>
{prompt}</s>
<|assistant|>
"""
    
    outputs = generator(
        chat_prompt,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.9,
        repetition_penalty=1.1,
        return_full_text=False
    )
    
    return outputs[0]['generated_text'].strip()

# Test it
print(generate_response("Say hello!", max_tokens=20))

### Part 2: Your Information

Fill in the cell below:

In [ ]:
# Student Information
STUDENT_NAME = "Your Name Here"
STUDENT_ID = "Your ID Here"

# Which option did you choose?
CHOSEN_OPTION = "A"  # Change to "A", "B", or "C"

print(f"Student: {STUDENT_NAME} ({STUDENT_ID})")
print(f"Chosen Option: {CHOSEN_OPTION}")

### Part 3: Test Data

Create your test dataset below. Examples are provided for each option - modify based on your choice.

In [ ]:
# ═══════════════════════════════════════════════════════
# OPTION A: Customer Support Tickets
# ═══════════════════════════════════════════════════════

support_tickets = [
    {
        "id": 1,
        "text": "My laptop won't turn on after the latest update. I've tried holding the power button but nothing happens. Model: XPS 15.",
        "expected_category": "Technical",
        "expected_product": "XPS 15",
    },
    {
        "id": 2,
        "text": "I was charged twice for my subscription this month. Please refund the duplicate charge. Order #12345.",
        "expected_category": "Billing",
        "expected_product": "Subscription",
    },
    {
        "id": 3,
        "text": "What are your store hours on weekends? Also, do you offer gift wrapping?",
        "expected_category": "General",
        "expected_product": None,
    },
    {
        "id": 4,
        "text": "This is the THIRD time my order arrived damaged! I want a full refund and compensation. Terrible service!",
        "expected_category": "Complaint",
        "expected_product": None,
    },
    {
        "id": 5,
        "text": "How do I reset my password? I can't log into my account.",
        "expected_category": "Technical",
        "expected_product": "Account",
    },
    # ADD MORE TICKETS HERE (need at least 15 total)
    # {
    #     "id": 6,
    #     "text": "...",
    #     "expected_category": "...",
    #     "expected_product": "...",
    # },
]

print(f"Option A: {len(support_tickets)} support tickets loaded")

In [ ]:
# ═══════════════════════════════════════════════════════
# OPTION B: News Articles
# ═══════════════════════════════════════════════════════

news_articles = [
    {
        "id": 1,
        "title": "Tech Giants Report Record Earnings",
        "text": """Apple, Microsoft, and Google parent Alphabet all reported 
        better-than-expected quarterly earnings on Tuesday. Apple's revenue 
        rose 8% to $94.8 billion, driven by strong iPhone sales. CEO Tim Cook 
        expressed optimism about the company's AI initiatives.""",
        "expected_topic": "Technology",
        "expected_entities": ["Apple", "Microsoft", "Google", "Tim Cook"],
    },
    {
        "id": 2,
        "title": "City Council Approves New Budget",
        "text": """The Springfield City Council voted 7-2 on Wednesday to approve 
        a $500 million budget for fiscal year 2025. Mayor Jane Smith praised 
        the bipartisan cooperation. The budget includes funding for new schools 
        and road repairs.""",
        "expected_topic": "Politics",
        "expected_entities": ["Springfield", "Jane Smith"],
    },
    {
        "id": 3,
        "title": "Lakers Defeat Celtics in Overtime Thriller",
        "text": """LeBron James scored 42 points as the Los Angeles Lakers 
        defeated the Boston Celtics 118-115 in overtime at Staples Center. 
        The victory extends the Lakers' winning streak to 5 games.""",
        "expected_topic": "Sports",
        "expected_entities": ["LeBron James", "Los Angeles Lakers", "Boston Celtics"],
    },
    # ADD MORE ARTICLES HERE (need at least 5 total)
]

print(f"Option B: {len(news_articles)} news articles loaded")

In [ ]:
# ═══════════════════════════════════════════════════════
# OPTION C: Recipes
# ═══════════════════════════════════════════════════════

recipes = [
    {
        "id": 1,
        "name": "Chocolate Chip Cookies",
        "text": """Mix 2 cups flour, 1 cup butter, 1 cup sugar, 2 eggs, 
        1 tsp vanilla, 1 tsp baking soda, and 2 cups chocolate chips. 
        Bake at 350°F for 10-12 minutes.""",
        "servings": 24,
        "expected_ingredients": ["flour", "butter", "sugar", "eggs", "vanilla", "baking soda", "chocolate chips"],
    },
    {
        "id": 2,
        "name": "Simple Pasta",
        "text": """Boil 1 lb pasta. In a pan, sauté 4 cloves garlic in 
        3 tbsp olive oil. Add 1 can crushed tomatoes, salt and pepper. 
        Simmer 15 minutes. Toss with pasta and top with parmesan.""",
        "servings": 4,
        "expected_ingredients": ["pasta", "garlic", "olive oil", "tomatoes", "parmesan"],
    },
    # ADD MORE RECIPES HERE
]

print(f"Option C: {len(recipes)} recipes loaded")

### Part 4: Implement Your NLP Tasks

Implement at least **2 NLP tasks** for your chosen option.

#### Task 1: Primary Classification/Extraction Task

In [ ]:
# ═══════════════════════════════════════════════════════
# TASK 1: YOUR PRIMARY NLP TASK
# ═══════════════════════════════════════════════════════

# Document your prompt engineering choices here:
# - Why did you structure the prompt this way?
# - Did you use zero-shot or few-shot?
# - What temperature did you choose and why?

"""
PROMPT ENGINEERING NOTES:
========================
[Write your notes here]

"""

def task1_function(text):
    """
    YOUR TASK 1 IMPLEMENTATION
    
    For Option A: Classify ticket category
    For Option B: Classify article topic
    For Option C: Extract ingredients
    """
    
    # YOUR PROMPT HERE
    prompt = f"""
    [Design your prompt here]
    
    Text: {text}
    
    [Expected output format]
    """
    
    response = generate_response(prompt, max_tokens=50, temperature=0.3)
    
    # YOUR POST-PROCESSING HERE
    result = response  # Process the response as needed
    
    return result

# Test Task 1
print("Testing Task 1...")
# test_result = task1_function("your test text here")
# print(f"Result: {test_result}")

#### Task 2: Secondary NLP Task

In [ ]:
# ═══════════════════════════════════════════════════════
# TASK 2: YOUR SECONDARY NLP TASK
# ═══════════════════════════════════════════════════════

"""
PROMPT ENGINEERING NOTES:
========================
[Write your notes here]

"""

def task2_function(text):
    """
    YOUR TASK 2 IMPLEMENTATION
    
    For Option A: Extract product/issue info
    For Option B: Extract named entities
    For Option C: Scale ingredients
    """
    
    prompt = f"""
    [Design your prompt here]
    
    Text: {text}
    """
    
    response = generate_response(prompt, max_tokens=100, temperature=0.3)
    
    return response

# Test Task 2
print("Testing Task 2...")
# test_result = task2_function("your test text here")
# print(f"Result: {test_result}")

#### (Optional) Task 3: Additional Feature

In [ ]:
# ═══════════════════════════════════════════════════════
# TASK 3: OPTIONAL ADDITIONAL TASK
# ═══════════════════════════════════════════════════════

def task3_function(text):
    """
    OPTIONAL: Additional NLP task
    
    For Option A: Generate response template
    For Option B: Summarize article
    For Option C: Suggest substitutions
    """
    pass  # Implement if desired

#### Part 5: Run Tests and Collect Results

Run your implementation on all test cases and collect results.

In [ ]:
# ═══════════════════════════════════════════════════════
# RUN ALL TESTS
# ═══════════════════════════════════════════════════════

results = []

# Example for Option A (modify for your chosen option)
print("="*70)
print("RUNNING TESTS")
print("="*70)

# Select your test data based on chosen option
if CHOSEN_OPTION == "A":
    test_data = support_tickets
elif CHOSEN_OPTION == "B":
    test_data = news_articles
else:
    test_data = recipes

for item in test_data:
    print(f"\nProcessing item {item['id']}...")
    
    text = item.get('text', '')
    
    # Run LLM method
    llm_result = task1_function(text)
    
    # Store results
    results.append({
        'id': item['id'],
        'text_preview': text[:50] + '...',
        'llm_result': llm_result,
        'expected': item.get('expected_category', item.get('expected_topic', 'N/A')),
    })

print(f"\n✓ Processed {len(results)} test cases")

### Part 6: Results Summary

In [ ]:
# ═══════════════════════════════════════════════════════
# RESULTS TABLE
# ═══════════════════════════════════════════════════════

import pandas as pd

df = pd.DataFrame(results)
print("\nRESULTS SUMMARY:")
print("="*70)
print(df.to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════
# ACCURACY CALCULATION
# ═══════════════════════════════════════════════════════

# Calculate accuracy for both methods
llm_correct = sum(1 for r in results if r['llm_result'] == r['expected'])
total = len(results)

print("\nACCURACY COMPARISON:")
print("="*50)
print(f"LLM Method:         {llm_correct}/{total} = {100*llm_correct/total:.1f}%")
print("="*50)